In [1]:
import subprocess
subprocess.run(["pip3", "install", "torch"], check=True)

Defaulting to user installation because normal site-packages is not writeable


You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


CompletedProcess(args=['pip3', 'install', 'torch'], returncode=0)

In [2]:
import torch
print(torch.__version__)

2.8.0


In [5]:
import pandas as pd
from collections import Counter

# ── Cargar dataset ───────────────────────────────────────────────────────
df = pd.read_csv("WELFake_clean.csv").dropna(subset=['text', 'label'])

# ── Split train/val/test ─────────────────────────────────────────────────
from sklearn.model_selection import train_test_split

X = df['text'].astype(str).tolist()
y = df['label'].tolist()

# Primero separas test (20%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Luego val (10% del total = 12.5% del temp)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.125, random_state=42, stratify=y_temp
)

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

# ── Vocabulario con 20,000 palabras SOLO del train ───────────────────────
contador = Counter()
for doc in X_train:
    contador.update(doc.lower().split())

# Tokens especiales + 20,000 más

Train: 44177 | Val: 6311 | Test: 12622


In [7]:
import re

def limpiar_texto(texto):
    texto = texto.lower()                            # minúsculas
    texto = re.sub(r'https?://\S+', '', texto)       # quita URLs
    texto = re.sub(r'[^a-z\s]', '', texto)           # quita puntuación y números
    texto = re.sub(r'\s+', ' ', texto).strip()       # quita espacios extra
    return texto

# Aplicar a los tres splits
X_train = [limpiar_texto(t) for t in X_train]
X_val   = [limpiar_texto(t) for t in X_val]
X_test  = [limpiar_texto(t) for t in X_test]

print("✅ Texto limpio")
print(f"Ejemplo: {X_train[0][:100]}")

✅ Texto limpio
Ejemplo: bias bashers more beer less vodka as russians mull ongoing crisis with the crisis continuing russian


In [8]:
from collections import Counter

# Vocabulario con 20,000 palabras SOLO del train
contador = Counter()
for doc in X_train:
    contador.update(doc.lower().split())

# Tokens especiales + 20,000 más frecuentes
vocab = ["<PAD>", "<UNK>"] + [word for word, _ in contador.most_common(20000)]
word2idx = {word: i for i, word in enumerate(vocab)}

print(f"Tamaño vocabulario: {len(vocab)}")  # debe ser 20,002

Tamaño vocabulario: 20002


In [9]:
def tokenize(text, word2idx, max_len=200):
    tokens = text.lower().split()[:max_len]          # trunca a 200
    indices = [word2idx.get(t, 1) for t in tokens]   # 1 = <UNK>
    # padding si es menor a 200
    indices += [0] * (max_len - len(indices))         # 0 = <PAD>
    return indices

# Verificación rápida
ejemplo = tokenize(X_train[0], word2idx)
print(f"Longitud secuencia: {len(ejemplo)}")          # debe ser 200
print(f"Primeros 10 índices: {ejemplo[:10]}")

Longitud secuencia: 200
Primeros 10 índices: [3063, 1, 47, 5529, 358, 12694, 17, 1522, 1, 1887]


In [10]:
import torch
from torch.utils.data import Dataset, DataLoader

# ── Dataset ──────────────────────────────────────────────────────────────
class FakeNewsDataset(Dataset):
    def __init__(self, texts, labels, word2idx, max_len=200):
        self.texts  = [tokenize(t, word2idx, max_len) for t in texts]
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        x = torch.tensor(self.texts[idx], dtype=torch.long)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y

# ── Crear datasets ───────────────────────────────────────────────────────
train_dataset = FakeNewsDataset(X_train, y_train, word2idx)
val_dataset   = FakeNewsDataset(X_val,   y_val,   word2idx)
test_dataset  = FakeNewsDataset(X_test,  y_test,  word2idx)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

# ── DataLoaders ──────────────────────────────────────────────────────────
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False)

print(f"Batches en train: {len(train_loader)}")

Train: 44177 | Val: 6311 | Test: 12622
Batches en train: 691


In [11]:
import urllib.request
import zipfile
import numpy as np
import os

# ── Descargar GloVe 6B 50d ───────────────────────────────────────────────
glove_url  = "https://nlp.stanford.edu/data/glove.6B.zip"
glove_zip  = "glove.6B.zip"
glove_file = "glove.6B.50d.txt"

if not os.path.exists(glove_file):
    print("Descargando GloVe... (puede tardar unos minutos)")
    urllib.request.urlretrieve(glove_url, glove_zip)
    print("Extrayendo...")
    with zipfile.ZipFile(glove_zip, 'r') as z:
        z.extract(glove_file)
    print("✅ GloVe listo")
else:
    print("✅ GloVe ya existe, saltando descarga")

# ── Cargar vectores GloVe ────────────────────────────────────────────────
print("Cargando vectores...")
glove_vectors = {}
with open(glove_file, 'r', encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word   = values[0]
        vector = np.array(values[1:], dtype=np.float32)
        glove_vectors[word] = vector

print(f"Vectores cargados: {len(glove_vectors)}")

# ── Matriz de embeddings (vocab_size x 50) ───────────────────────────────
vocab_size       = len(vocab)
embedding_dim    = 50
embedding_matrix = np.zeros((vocab_size, embedding_dim), dtype=np.float32)

cubiertos = 0
for word, idx in word2idx.items():
    if word in glove_vectors:
        embedding_matrix[idx] = glove_vectors[word]
        cubiertos += 1

print(f"Shape matriz: {embedding_matrix.shape}")  # (20002, 50)

# ── Tarea 8: Cobertura ───────────────────────────────────────────────────
cobertura = cubiertos / vocab_size * 100
print(f"Cobertura GloVe: {cobertura:.2f}%")

✅ GloVe ya existe, saltando descarga
Cargando vectores...
Vectores cargados: 400000
Shape matriz: (20002, 50)
Cobertura GloVe: 96.86%


In [12]:
import torch
import torch.nn as nn

# ── nn.Embedding con pesos GloVe ─────────────────────────────────────────
embedding_tensor = torch.tensor(embedding_matrix, dtype=torch.float)

embedding_layer = nn.Embedding(vocab_size, embedding_dim)
embedding_layer.weight = nn.Parameter(embedding_tensor)
embedding_layer.weight.requires_grad = False

print(f"Embedding layer: {embedding_layer}")

# ── Verificación — formas de un batch ───────────────────────────────────
batch_texts, batch_labels = next(iter(train_loader))

print(f"\nbatch_texts.shape:  {batch_texts.shape}")
print(f"batch_labels.shape: {batch_labels.shape}")
print(f"embedding_matrix.shape: {embedding_matrix.shape}")

batch_embedded = embedding_layer(batch_texts)
print(f"batch_embedded.shape: {batch_embedded.shape}")

Embedding layer: Embedding(20002, 50)

batch_texts.shape:  torch.Size([64, 200])
batch_labels.shape: torch.Size([64])
embedding_matrix.shape: (20002, 50)
batch_embedded.shape: torch.Size([64, 200, 50])
